# Fix: Label Agent 3 Training Data with LLM API

Steps:
1. Use DeepSeek API (ChatGPT as fallback) to classify patient text from chatbot dataset
2. Automatically discard samples that don't fit any of the 10 departments (SKIP)
3. Build Agent 3 training data using classification results
4. Verify department distribution + quick 50-step training + inference test

In [ ]:
!pip install -q transformers>=4.45.0 "trl==0.12.0" peft>=0.13.0 \
    bitsandbytes>=0.44.0 accelerate>=1.0.0 datasets openai

In [ ]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [ ]:
import shutil, os
from google.colab import drive
if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive')
drive.mount('/content/drive')
print('Google Drive mounted.')

## Step 1: Configure LLM API (DeepSeek primary, ChatGPT fallback)

In [ ]:
import time
from openai import OpenAI
from google.colab import userdata

VALID_DEPARTMENTS = {
    'Cardiology', 'Neurology', 'Dermatology', 'Gastroenterology',
    'Endocrinology', 'Pulmonology', 'Infectious Disease',
    'Orthopedics', 'Urology', 'General Medicine',
}
VALID_URGENCIES = {'Routine', 'Urgent', 'Emergency'}
DEPT_LIST = ', '.join(sorted(VALID_DEPARTMENTS))

# ── DeepSeek (primary) ──
# Add DEEPSEEK_API_KEY in Colab Secrets
deepseek_client = OpenAI(
    api_key=userdata.get('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com',
)

# ── ChatGPT (fallback) ──
# Add OPENAI_API_KEY in Colab Secrets
chatgpt_client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY'),
)

CLASSIFY_SYSTEM_PROMPT = f'''You are a medical triage classifier.
Given a patient's description, do TWO things:
1. Classify into exactly one department from: {DEPT_LIST}
   If the patient's issue does not fit ANY of these departments, output Department: SKIP
2. Assign urgency: Routine, Urgent, or Emergency

Respond in exactly this format (nothing else):
Department: <department or SKIP>
Urgency: <Routine|Urgent|Emergency>'''

print('API clients configured.')
print(f'Valid departments: {DEPT_LIST}')

In [ ]:
import re

def _call_llm(client, model, patient_text):
    """Call an LLM and return raw text."""
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': CLASSIFY_SYSTEM_PROMPT},
            {'role': 'user', 'content': patient_text},
        ],
        max_tokens=50,
        temperature=0.1,
    )
    return resp.choices[0].message.content.strip()


def classify_patient_text(patient_text):
    """DeepSeek first, fallback to ChatGPT.
    Returns (department, urgency) or (None, None) for SKIP.
    """
    text = None

    # Try DeepSeek
    try:
        text = _call_llm(deepseek_client, 'deepseek-chat', patient_text)
    except Exception as e:
        print(f'  [DeepSeek failed: {e}], trying ChatGPT...')

    # Fallback to ChatGPT
    if text is None:
        try:
            text = _call_llm(chatgpt_client, 'gpt-4o-mini', patient_text)
        except Exception as e:
            print(f'  [ChatGPT also failed: {e}]')
            return None, None

    # Parse response
    dept_match = re.search(r'Department:\s*([^\n]+)', text)
    urg_match = re.search(r'Urgency:\s*([^\n]+)', text)
    department = dept_match.group(1).strip() if dept_match else None
    urgency = urg_match.group(1).strip() if urg_match else 'Routine'

    # SKIP if not in valid department list
    if department is None or department == 'SKIP' or department not in VALID_DEPARTMENTS:
        return None, None
    if urgency not in VALID_URGENCIES:
        urgency = 'Routine'
    return department, urgency


# Quick sanity check
test_texts = [
    'I have itchy rash on my skin for a week',
    'I am experiencing severe chest pain and shortness of breath',
    'I have a terrible headache and dizziness',
    'I had 2 teeth removed on thursday',          # Should SKIP (no dentistry dept)
    'I have PCOD problem and also cyst',           # Should SKIP (no gynecology dept)
]
print('--- LLM API Sanity Check ---')
for t in test_texts:
    dept, urg = classify_patient_text(t)
    label = f'{dept} ({urg})' if dept else 'SKIP'
    print(f'  "{t[:60]}" → {label}')

## Step 2: Label chatbot data with LLM API, build Agent 3 training data

In [ ]:
import json, random
from datasets import load_dataset as dl

# Load doctor schedules
SCHEDULE_PATH = '/content/drive/MyDrive/doctor_schedules.json'
# Upload if not on Drive
if not os.path.exists(SCHEDULE_PATH):
    from google.colab import files
    print('Please upload doctor_schedules.json')
    uploaded = files.upload()
    SCHEDULE_PATH = 'doctor_schedules.json'

with open(SCHEDULE_PATH) as f:
    schedules = json.load(f)

schedules_by_dept = {}
for entry in schedules:
    schedules_by_dept.setdefault(entry['department'], []).append(entry)
print(f'Loaded {len(schedules)} schedule entries, {len(schedules_by_dept)} departments')

In [ ]:
INSTRUCTION = (
    'You are a compassionate medical assistant. '
    'A patient has been assigned an appointment. '
    'Write a warm, clear appointment confirmation and practical pre-visit instructions. '
    'Keep the tone professional but reassuring. '
    'Format your response as:\n'
    'Confirmation: <one sentence confirming the appointment>\n'
    'Instructions: <2-4 specific pre-visit instructions>'
)

MIN_RESPONSE_LEN = 80
MAX_RESPONSE_LEN = 800
MAX_SYMPTOM_LEN = 200

def make_record(patient_q, doctor_a, department, urgency):
    """Build a training record using LLM API classification result."""
    doctor_a = str(doctor_a).strip()
    if len(doctor_a) < MIN_RESPONSE_LEN or len(doctor_a) > MAX_RESPONSE_LEN:
        return None

    dept_scheds = schedules_by_dept.get(department,
                      schedules_by_dept.get('General Medicine', schedules))
    entry = random.choice(dept_scheds)
    doctor = entry['doctor']
    time_slot = f"{entry['day']} at {entry['time_slot']}"
    symptoms = str(patient_q).strip()[:MAX_SYMPTOM_LEN]

    formatted_output = (
        f'Confirmation: Your appointment with {doctor} in {department} '
        f'has been confirmed for {time_slot}. '
        f"We're here to help with your concerns.\n"
        f'Instructions: {doctor_a}'
    )

    return {
        'instruction': INSTRUCTION,
        'input': (f'Patient symptoms: {symptoms}\n'
                  f'Assigned department: {department}\n'
                  f'Doctor: {doctor}\n'
                  f'Appointment: {time_slot}\n'
                  f'Urgency: {urgency}'),
        'output': formatted_output,
        'department': department,
        'urgency': urgency,
    }

In [ ]:
# ============================================================
# Small sample test: 500 records
# Change to None for full run (20% sample) after verification
# ============================================================
SAMPLE_SIZE = 500  # Set to None for full run (20% sample)

random.seed(42)
ds = dl('ruslanmv/ai-medical-chatbot', split='train')
print(f'Total rows: {len(ds)}')

n_sample = SAMPLE_SIZE if SAMPLE_SIZE else int(len(ds) * 0.20)
indices = random.sample(range(len(ds)), n_sample)
sampled = ds.select(indices)
print(f'Sampled: {len(sampled)} rows')

In [ ]:
# Classify each record with LLM API and build training records
records = []
dept_counter = {}
skipped = 0

for i, row in enumerate(sampled):
    patient_text = row.get('Patient', row.get('input', ''))
    doctor_text = row.get('Doctor', row.get('output', ''))

    # LLM API classification
    department, urgency = classify_patient_text(str(patient_text).strip()[:500])

    # SKIP: doesn't fit any of the 10 departments
    if department is None:
        skipped += 1
        if (i + 1) % 50 == 0:
            print(f'  [{i+1}/{len(sampled)}] kept={len(records)}, skipped={skipped}')
        continue

    rec = make_record(patient_text, doctor_text, department, urgency)
    if rec:
        records.append(rec)
        dept_counter[department] = dept_counter.get(department, 0) + 1

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/{len(sampled)}] kept={len(records)}, skipped={skipped}')

print(f'\nTotal records: {len(records)}')
print(f'Skipped (not in 10 departments): {skipped}')

# Department distribution
print('\n--- Department Distribution ---')
for dept, cnt in sorted(dept_counter.items(), key=lambda x: -x[1]):
    print(f'  {dept:<25} {cnt:>4}  ({cnt/len(records):.1%})')

# Preview 5 samples
print('\n--- Sample Records ---')
for r in records[:5]:
    lines = r['input'].split('\n')
    print(f'  {lines[0][:80]}')
    print(f'  {lines[1]}')
    print()

## Step 2 Checkpoint
- Department distribution should be spread across multiple departments, not all General Medicine
- Sample records should show correct symptom-to-department mapping
- Skipped count should be reasonable (dental, gynecology, ophthalmology cases correctly discarded)

If everything looks good, continue to Step 3

## Step 3: Quick 50-step training + inference test

In [ ]:
# No need to free Agent 1 VRAM (using API instead)
# All GPU memory available for Agent 3 training
import torch
print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def format_chat(example):
    messages = [
        {'role': 'system', 'content': example['instruction']},
        {'role': 'user', 'content': example['input']},
        {'role': 'assistant', 'content': example['output']},
    ]
    return {'text': tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)}

random.shuffle(records)
split = int(len(records) * 0.85)
train_ds = Dataset.from_list(records[:split]).map(
    format_chat, remove_columns=list(records[0].keys()))
test_ds = Dataset.from_list(records[split:]).map(
    format_chat, remove_columns=list(records[0].keys()))
print(f'Train: {len(train_ds)} | Test: {len(test_ds)}')

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto',
    torch_dtype=torch.bfloat16)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', task_type='CAUSAL_LM'))
model.print_trainable_parameters()

In [ ]:
args = SFTConfig(
    output_dir='/content/test_output',
    max_steps=50,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    max_seq_length=512,
    dataset_text_field='text',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    report_to='none',
)

trainer = SFTTrainer(
    model=model, args=args,
    train_dataset=train_ds, processing_class=tokenizer)

trainer.train()
print('Quick training done!')

In [ ]:
# Inference test: 5 samples
model.eval()
test_samples = records[split:split+5]
print('=== Inference Test ===\n')

for i, sample in enumerate(test_samples):
    messages = [
        {'role': 'system', 'content': sample['instruction']},
        {'role': 'user', 'content': sample['input']},
    ]
    tokenized = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)

    input_len = tokenized['input_ids'].shape[-1]
    with torch.inference_mode():
        output = model.generate(**tokenized, max_new_tokens=200,
                                temperature=0.7, do_sample=True, top_p=0.9,
                                pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

    input_dept = sample['department']
    output_has_dept = input_dept.lower() in generated.lower()
    status = 'MATCH' if output_has_dept else 'MISMATCH'

    print(f'[{i+1}] {status}')
    print(f'    Input dept:  {input_dept}')
    print(f'    Output:      {generated[:200]}...')
    print()

## Results Checklist

**Step 2:**
- Department distribution is spread (not all General Medicine)
- Symptoms match assigned departments

**Step 3:**
- Confirmation output includes the correct department from input

**If all good:**
1. Change `SAMPLE_SIZE = 500` to `SAMPLE_SIZE = None` for full run
2. Save train/test jsonl to Drive
3. Use `train_response_generator.ipynb` for full training